# `C03` — Price-time priority, ties, and ownership blindness

Scenario **C03** of the Track A certification ladder (card `OMIE-CERT-C03-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C3). Oracle kind: **exact-parametric, plus M1 and M6**.

**Round r2** (`OMIE-CERT-RERUN-1`, REPEAT at K2, correction entry 19). Sections 1–5 and 7 are re-authored from the specification to add **same-second assertions**: the stored stamps of orders entered in the same second must be unique and sequential. In r1 every asserted tape had strictly increasing tape seconds, so the stamp claim held by construction of the dataset alone.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. Every pass/fail assertion here is an **exact integer equality** on order ids, prices in ticks (0.01 EUR/MWh) and quantities in ticks (0.1 MW); euro figures are shown, never asserted.

## 1. Purpose

**Purpose.** Fairness of the queue is what makes a limit order book a market rather than a lottery,
and it is the rule an agent's passive orders live or die by. (OQ-25 once traced the agent's 0.63 %
passive fill rate to its queue position behind never-cancelled phantom liquidity. That attribution is
**no longer supported**: until `OMIE-AGENT-TIME-PRIORITY-1`, the environment stamped agent orders from
its minutes clock, which put them at the *front* of every same-price queue. The figure was therefore
measured with front-of-queue priority, and it is flagged for re-measurement — see
[the note in Track B](../cid_rl/scenario-ladder-track-b.md#before-track-b-the-rl-model-is-simplified).) The scenario's important
design choice is **what it asserts**: not traded volume, but *which specific orders filled*, by order
ID. A priority bug leaves total volume unchanged, so a volume assertion would pass over it silently.
Decoys at worse prices are included in every draw so price ranking and time ranking are tested
together.

Two further claims ride along. §5.2 requires that all timestamps come from **a single global, unique,
sequential queue** — the property this scenario's entire tie-break claim rests on — so a monotonicity
and uniqueness assertion on the timestamp counter is folded in here. And §6.6's absence of self-trade
prevention becomes a positive assertion: a crossing pair carrying the **same** agent identifier must
print a trade, identical in price and volume to the distinct-agent counterfactual. That is relation
M6, and it costs one cell.

*(Verbatim from `scenario-ladder.md` §C3. Its Track B link points to a research document that is not part of this release.)*

**Certifies** (card): §6.1 price-time priority; §5.2 global unique sequential timestamps, **including among orders entered in the same second** (r2); §6.6 no self-trade prevention; §11.1 row *"Price-time priority"*.

**How the claim is pinned.** A draw is a queue of resting orders on one side, then one aggressor on the other side. The $k \in [2,6]$ *touch* orders share the best price, carry the same size $q$, and enter one after another; one to three *decoys* sit at worse prices. The aggressor is sized $j \cdot q$ with $1 \le j < k$. Per run the notebook asserts, by order id:

1. **the filled set**: exactly the $j$ earliest touch orders (the card's pass criterion: set equality);
2. **the counterparty sequence**: the exact ordered list of fills (passive id, aggressor id, price, quantity), so a queue that takes the right orders in the wrong order also fails (domain review §17.2, D3-08: pin the exact counterparty sequence);
3. **the book afterwards**: the other $k - j$ touch orders and every decoy still rest, untouched, at their full size;
4. **the stored timestamps** (§5.2): the engine's own timestamps on the resting orders, read from the book just before the aggressor enters, are unique and strictly increasing in entry order, on every run;
5. **the same-second stamps** (§5.2 with §13.1; r2): on the runs where several resting orders share one tape second, the stamps inside each such second are unique and sequential in entry order. This is recorded separately among same-price orders and across prices, because §5.2's queue is global. On those runs (1)–(3) are asserted exactly as well, so a same-price tie inside one second must go to the order entered first.

**Why same-second runs (r2).** A submission time is a whole second (§13.1, `HH:MM:SS`), and real sessions put many orders in one second. A stamp that merely copied the tape's second would be unique on every r1 tape and would still pass (4). It fails only where two orders share a second, so r2 builds such tapes.

**Four ways a wrong ranker is made to fail.**
- *Time before price* (a FIFO queue that ignores price): `C03.price_over_time` enters the decoys **first** and gives the aggressor a limit that reaches every decoy, so a time-only ranker fills decoys on every draw. In `C03.priority_ladder` the decoys enter after the touch orders.
- *Latest first* (LIFO at a price level): fills the $j$ latest touch orders. Since $j < k$, that set always differs.
- *Order id instead of time*: on a generator tape the order ids grow with entry order, so an id ranker is indistinguishable from a time ranker. The **M1 reversal** run, made from every draw, re-enters the touch orders in reverse order (the timestamp column is rewritten so it still grows down the tape). Price-time priority now fills the touch orders with the **highest** ids; an id ranker still fills the lowest.
- *Stamps copied from the tape's second*: on the same-second runs (section 4) several resting orders share one second, so copied stamps are equal and (5) fails. The reversed same-second runs also make the tie decide the fills: price-time priority takes the touch orders entered first, which there carry the **highest** ids. A book that breaks equal stamps by order id takes the lowest.

**M6** (§6.6): the `C03.same_owner_pair` draws give both legs of a crossing pair the **same** agent id. The pair must trade, and its trade list must equal, entry for entry, that of the distinct-owner counterfactual the generator ships with it. The first `SELF_MATCH_DRAWS` ladder draws of each construction add a second M6 run in which the aggressor takes the touch orders' agent id, so every fill is a self-match.

**K2 boundary twins** (section 4) change only the aggressor's size, or only the entry order and timestamp column: one tick over and one tick under $j \cdot q$, the whole touch level, and a walk into the best decoy. Two more twins give the touch orders **one shared tape second**, in the original and in reversed entry order (relation M1 as the battery states it). In r1 these two were asserted only up to the tie order; r2 asserts them exactly. The **same-second runs** (r2) put **every** row of the tape in one second, in the original and the reversed touch order, on the first `SAME_SECOND_DRAWS` valid draws of each ladder construction.

**Does not certify.**
- That survivors keep their queue order after a cancellation: C04.
- The timestamp and later priority of a partially executed order (§5.2's partial-execution bullet): C05. The `one_tick_over` twin only reads the residual's quantity.
- Volume-weighted prices over several levels: C06. Iceberg slices and their new timestamps (§5.2; review D3-08): C08 and C09.
- The arrival order **inside one second of real replay data**. OMIE's files record whole seconds (§13.1), and the order in which the simulator processes same-second rows of a real file is a modelling choice (OQ-32). On a synthetic tape the row order **is** the submission order, so C03 asserts against row order.
- The stamp values themselves: which second, and how the sub-second part is formed. Only their order and uniqueness are asserted, as §5.2 states them.
- The aggressor's own stamp: (4) and (5) read the book just before it enters. Explicit capacity requests (§5.2's other user of the queue) do not exist in the single-zone simulator (§6.3).
- The trade timestamp (§8 / Matching Algorithm §5.3). It is compared only between the two runs of an M6 pair.
- Anonymity of what agents observe (§7): C13. Ownership blindness on random sessions at scale: C15.

## 2. The rule, quoted

The governing passages, quoted **verbatim** from [`market-spec.md`](../../docs/omiecid/market-spec.md) and checked against it at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. The regulatory sources behind §5.2 and §6.1 are shown as citations, not machine-checked.

How the quotes ground the expectation of section 5:

- **Who rests (§4.5, §6.2 item 1).** Every row before the aggressor is on one side, so it meets no counterpart on entry. A NON order that does not match enters the book (§4.5, *Book entry if unmatched: Yes*). The last row is the incoming order of §6.2 item 1.
- **In what order the aggressor takes them (§6.1, §11.1).** On the resting side, best price first, then earliest timestamp. Each fill is followed by the next-best counterpart (§6.2 item 1 again), and a level that runs out hands over to the next (§6.2 item 6, reached only by two twins). Matching stops when the aggressor is exhausted or no counterpart crosses (§6.1's $\ge$).
- **What "earliest" means (§5.2).** An order receives its timestamp when it enters the SOB, from one global queue, unique and sequential. Entry is submission, so the earlier-submitted of two equal-priced orders holds the earlier timestamp and goes first. The same rule makes the engine's stored timestamps unique and strictly increasing in entry order, and this is asserted directly.
- **Inside one second (§5.2, §13.1; r2).** A submission time is a whole second (§13.1: `DD/MM/YYYY HH:MM:SS`), so several orders can enter in the same second, and a tape that gives them one timestamp value is an ordinary tape. §5.2's queue is not that clock. It still hands them unique, sequential timestamps in the order they enter, and the Matching Algorithm states the consequence (§5.5, cited below): "it is always possible to sort them by time stamp unequivocally". A tape enters its rows in row order. So inside a second the earlier row holds the earlier stamp and wins a price tie. The queue is global, so the stamps are unique across prices too, not only among orders at one price.
- **At what price, for what quantity (§6.2 item 3; §4.5).** Each fill prints at its own maker's limit (C02 certifies this; it is re-asserted here per fill). Its quantity is the smaller of the aggressor's remainder and the maker's remaining size (derived as in C02's finding F1). The touch orders all have size $q$ and the aggressor $j \cdot q$, so each of the $j$ fills is a whole order.
- **Ownership (§6.6).** No exclusion by owner: a same-owner crossing pair trades, and relabelling owners cannot change a trade.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- §6.1: the ranking rule C03 certifies ----------------------------------------------------
    SpecQuote(
        "market-spec.md", "§6.1",
        "**Buy side:** ranked highest price first; ties broken by earliest timestamp.",
        note="Ranking, buy side: price first, then time. Exercised by every draw whose resting side is BUY.",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "**Sell side:** ranked lowest price first; ties broken by earliest timestamp.",
        note="Ranking, sell side: the mirror rule. Exercised by every draw whose resting side is SELL.",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "A match occurs when best bid price ≥ best ask price.",
        note="The aggressor crosses the touch on every draw; in C03.price_over_time it also reaches every decoy.",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "This is a hard rule mandated by ACER Decision 11/2024 Art. 5 and CACM Regulation Art. 51.",
        note="Provenance of the ranking rule (ACER/CACM source price-time priority; C02's maker-pricing row is "
             "sourced to the Matching Algorithm instead).",
    ),
    SpecQuote(
        "market-spec.md", "§11.1",
        "| Price-time priority for matching | ACER Decision 11/2024 Art. 5; CACM Regulation Art. 51 |",
        note="§11.1 lists price-time priority as a hard rule: the row the card certifies.",
    ),
    # --- §6.2: how one incoming order walks the ranked queue -------------------------------------
    SpecQuote(
        "market-spec.md", "§6.2 item 1",
        "Incoming order attempts to match against best counterpart in the consolidated order book.",
        note="The aggressor (the last row) is the incoming order; 'best' is §6.1's ranking, applied again after each fill.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 3",
        "Trade executes at the **resting order's limit price** (maker pricing).",
        note="Each fill prints at its own maker's limit: p_touch on the touch, a decoy's limit in the into_decoys twin.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 6",
        "If matching exhausts one price level, continues to next best price level.",
        note="Only the all_touch and into_decoys twins reach past the touch; a drawn aggressor stops inside it.",
    ),
    # --- §5.2: the single global timestamp queue the tie-break rests on --------------------------
    SpecQuote(
        "market-spec.md", "§5.2",
        "Each order receives a timestamp when entering the SOB.",
        note="The timestamp is assigned at entry, so 'earliest timestamp' is 'entered first', also when two "
             "orders enter in the same second (r2).",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "All timestamps across orders and explicit capacity requests are assigned from a single global queue "
        "(unique, sequential). Source: Continuous Trading Matching Algorithm §4.3.",
        note="Asserted on the engine's stored timestamps of the resting orders on every run: unique, and strictly "
             "increasing in entry order. r2 also asserts it inside each second that several orders share, "
             "at one price and across prices (the queue is global).",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "Likewise a resting order that is partially executed keeps its timestamp.",
        note="Context for the one_tick_over twin only: the partially executed touch order stays in the book with "
             "its residual. Its timestamp and later priority are C05's, not asserted here.",
    ),
    # --- §13.1: submission times are whole seconds, so orders can share one (r2) ------------------
    SpecQuote(
        "market-spec.md", "§13.1",
        "| 13 | Momento de envío | Submission time | datetime | `01/06/2025 17:41:12` | DD/MM/YYYY HH:MM:SS format.",
        note="A submission time has one-second resolution, so a tape whose rows share one timestamp value is "
             "an ordinary tape. The same-second runs build exactly that.",
    ),
    # --- §6.6: no self-trade prevention (relation M6) --------------------------------------------
    SpecQuote(
        "market-spec.md", "§6.6",
        "The XBID/M7 matching engine does **not** implement self-trade prevention. Matching is pure price-time "
        "priority; an agent's buy order can match its own sell order, and orders from two different units of the "
        "same agent can match against each other.",
        note="An agent's buy matching its own sell is correct behaviour: the same-owner pair must trade.",
    ),
    SpecQuote(
        "market-spec.md", "§6.6",
        "**Simulator implication:** The matching engine must not reject or skip matches based on order ownership.",
        note="M6: relabelling owners, including into a self-match, leaves the trade list unchanged.",
    ),
    # --- who rests ---------------------------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§4.5",
        "| Restriction | Partial fill | Book entry if unmatched | Applicable to |",
        note="Header of the execution-restriction table, quoted so the NON row below can be read.",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| NON | Yes | Yes | Regular, Block |",
        note="Every resting row meets no counterpart on entry and enters the book; an untouched order stays there in full.",
    ),
)

_MA = ("docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/"
       "Continuous_Trading_Matching_Algorithm_Public_Description.md")
CITATIONS = (
    Citation(
        "Continuous Trading Matching Algorithm (public description) §4.3 Timestamp — " + _MA,
        "When an order or an explicit capacity request is entered into the SOB (orders) or CMM (explicit "
        "requests), it receives a timestamp. [...] All timestamps are assigned sequentially",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §4.4 Price-Time-Priority principle — same file",
        "Orders are ranked per side (buy/sell) best price first, meaning that buy orders are ranked from high est "
        "to lowest price and sell orders are ranked from lowest to highest price. Orders with the same price limit "
        "are prioritized by their time stamp oldest first.",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §5.5 Priority setting between implicit and "
        "explicit allocation — same file",
        "Both order entry (and modification, activation, and activation of a new slice of an iceberg order) events "
        "and explicit capacity requests enter the XBID system through a single queue and get their timestamp "
        "assigned in this single queue. As a result, all orders and all explicit capacity requests have unique "
        "timestamps, so that it is always possible to sort them by time stamp unequivocally.",
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"quotes: {sum(c.ok for c in QUOTE_CHECKS)}/{len(QUOTE_CHECKS)} match their source")

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`** valid draws of `C03.priority_ladder` and **`POT_DRAWS = 500`** of `C03.price_over_time` (the card's default for a random scenario, per construction), plus **`M6_DRAWS = 100`** of `C03.same_owner_pair` (the card's "separate M6 draw"; each one runs two tapes). Every valid draw of the three enters the draw accounting, so the run requests `DRAWS + POT_DRAWS + M6_DRAWS` valid draws.
- **`SEED0 = 0`**, the first seed of each construction. A draw found outside the capacity margin before the run, or marked I8 `invalid` by the battery after it, is **discarded and replaced by the next seed**. Each construction's loop stops at its draw count or after twice that many attempts, and the discard rate is reported.
- **`EAGER_DRAWS = 5`.** The first `EAGER_DRAWS` **valid** draws of each construction also run eager and are compared, integer field by integer field, with their jitted run (`compare_records`). On top of these, eager runs are also forced for: the first valid draw of each class not yet covered (construction × resting side; for the pair, aggressor side × size order); the first M1-reversal run and the first self-match run of each ladder construction; the first run of each boundary twin per ladder construction; and the first run of each same-second variant per ladder construction. Every one of these **must** have its eager comparison. A missing one is recorded as a failed `eager_jit` assertion. Because the eager runs are chosen among valid draws, a discarded seed cannot remove them.
- **`TWIN_DRAWS = 5`.** The first `TWIN_DRAWS` valid draws of each ladder construction also run the six boundary twins of section 4.
- **`SAME_SECOND_DRAWS = 100`** (r2). The first `SAME_SECOND_DRAWS` valid draws of each ladder construction also run the two same-second runs of section 4, where every row of the tape shares one second.
- **`SELF_MATCH_DRAWS = 100`.** The first `SELF_MATCH_DRAWS` valid draws of each ladder construction also run the M6 self-match relabelling.
- The M1 reversal runs on **every** valid ladder draw: it is the run that separates time priority from order-id order.
- The tick grids are **not** parameters: they come from `OMIEProductConfig()` and serve only to display euros. Every assertion is in integer ticks and order ids.

A run with far fewer draws than the default (say, under 30) can miss a value of $k$, of the decoy count, or a class. It then fails the coverage checks and the missing class's eager comparison. That is intended: such a run does not cover the card's dataset.

**Judging rules carried forward (binding, `OMIE-CERT-RERUN-1`).** A battery or relation status of `not_applicable` counts as satisfied, never as a failure. Every recorded list is recorded in full, never cut to its first few entries. Eager comparisons run on the first **valid** draws.

In [ ]:
CERT_ID = "C03"         # C00-C17 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500             # valid draws of C03.priority_ladder (card: random scenarios default to 500 draws)
POT_DRAWS = 500         # valid draws of C03.price_over_time (decoys entered first, worse-priced, reachable)
M6_DRAWS = 100          # valid draws of C03.same_owner_pair (the card's separate M6 draw)
SEED0 = 0               # first seed of each construction
EAGER_DRAWS = 5         # the first EAGER_DRAWS valid draws of each construction also run eager (plus forced minimums)
TWIN_DRAWS = 5          # the first TWIN_DRAWS valid draws of each ladder construction also run the boundary twins
SAME_SECOND_DRAWS = 100 # the first SAME_SECOND_DRAWS valid draws of each ladder construction also run the same-second runs
SELF_MATCH_DRAWS = 100  # the first SELF_MATCH_DRAWS valid draws of each ladder construction also run the M6 self-match

In [ ]:
import pathlib

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side

LADDER = "C03.priority_ladder"
POT = "C03.price_over_time"
PAIR = "C03.same_owner_pair"
LADDER_KEYS = (LADDER, POT)
CONSTRUCTIONS = (LADDER, POT, PAIR)
N_DRAWS = {LADDER: DRAWS, POT: POT_DRAWS, PAIR: M6_DRAWS}
REQUESTED = DRAWS + POT_DRAWS + M6_DRAWS

CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE
BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, ICE = int(ExecRestriction.NON), int(ExecRestriction.ICE)

SLICE_MARGIN = 64             # capacity-validity precondition: every iceberg has ceil(total/peak) <= 64
PRICE_INT_MAX = 1_999_800     # §2.3: +9,999 EUR/MWh in the offset encoding (0 is -9,999)
QTY_INT_MIN, QTY_INT_MAX = 1, 9_990   # §2.3: 0.1 MW and 999 MW in quantity ticks
K_RANGE = tuple(range(2, 7))  # card: k in [2, 6] touch orders
DECOY_RANGE = (1, 2, 3)       # the generator's decoy count (n_decoys / n_early)

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C03_priority_ties_and_ownership.ipynb"

print(f"{CERT_ID}: {LADDER} x {DRAWS}, {POT} x {POT_DRAWS}, {PAIR} x {M6_DRAWS} valid draws "
      f"(requested {REQUESTED}), seeds from {SEED0}, at most 2x attempts per construction")
print(f"eager: first {EAGER_DRAWS} valid draws per construction + forced minimums; twins on the first "
      f"{TWIN_DRAWS}; same-second runs on the first {SAME_SECOND_DRAWS}; self-match on the first "
      f"{SELF_MATCH_DRAWS}; M1 reversal on every ladder draw")
print(f"price tick {CONFIG.price_tick} EUR/MWh, qty tick {CONFIG.qty_tick} MW (display only)")
print(f"capacity envelope: {ENVELOPE}; slice margin <= {SLICE_MARGIN} per iceberg")

## 4. Dataset

**`C03.priority_ladder`** (`omie_tape_synth.make_priority_ladder`, `variant="ladder"`) draws, per seed: $k \in [2,6]$, $j \in [1, k-1]$, a decoy count in $[1,3]$, the resting side (BUY or SELL with probability ½), a touch price on the grid in [30, 90] EUR/MWh and one size $q$ on the grid in [2, 20] MW. It emits, all NON on one contract, with order ids and tape timestamps $1, 2, 3, \dots$ in row order: **the $k$ touch orders** (agent 1) at the touch price, each for $q$; then **the decoys** (agent 1), the $d$-th one $0.50\,d$ EUR/MWh worse than the touch, each for $q$; then **the aggressor** (agent 2) on the other side, 1.00 EUR/MWh through the touch, for $j \cdot q$. The aggressor's limit reaches the first two decoy levels, but its size stops it inside the touch.

**`C03.price_over_time`** (`variant="price_over_time"`) draws the same $k$, $j$, side, touch price and $q$, plus a step $s \in [1,50]$ ticks. It enters **the decoys first**: decoy $d$ (for $d = 1, \dots, n_{early}$, $n_{early} \in [1,3]$) is $d \cdot s$ ticks worse than the touch. Then come the $k$ touch orders, then the aggressor for $j \cdot q$, with its limit **at the deepest decoy's price**, so every decoy crosses it. A ranker that takes the earliest order first fills decoys; price-time priority fills none.

**`C03.same_owner_pair`** (`make_same_owner_pair`) is `C02.crossing_pair`'s draw for the same seed (one resting NON order, then a NON aggressor priced $\Delta \in [1,50]$ ticks through it, sizes in [1, 60] MW), with **both rows' agent id set to 7**. The generator ships the distinct-owner original (agents 1 and 2) as `aux_tapes["counterfactual"]`: same rows, same order ids, only the agent column differs.

**What section 5 reads.** The expectation reads the **tape rows**: their order, sides, prices, quantities and order ids. It reads the timestamp column only to find the rows that share one second (assertion (T=)); the ranking itself uses row order. `draw.params` is used for display, for the closed-form cross-check of section 5, and for a consistency check against the tape. `draw.expected` is never read.

**The precondition, checked from the tape for every valid draw** (an aggregate dataset assertion in section 7). For both ladder constructions: every row NON, one contract, prices and quantities within §2.3's bounds, distinct order ids, tape timestamps strictly increasing down the tape; every row but the last on one side and the last on the other; $k \in [2,6]$ rows at the best resting price, all of size $q$; every other resting row strictly worse (a decoy), with 1 to 3 decoys; the aggressor crossing the touch, sized $j \cdot q$ with $1 \le j < k$; in the ladder every decoy entered after every touch order; in price-over-time every decoy entered before every touch order **and** crossing the aggressor's limit; and `draw.params` equal to the tape's integers. For the pair: two NON rows, opposite sides, crossing; one agent id on both rows; and the counterfactual equal to the tape except in the agent column, where its two agent ids differ.

**The capacity margin** (scenario-ladder-plan, *The capacity-validity precondition*, five ceilings). At most 10 NON orders and no icebergs sit far inside it (at most 9 resting per side, at most 10 trade units, a hop bound of at most 12). It is still checked **before** each run, on draws and twins alike: a tape whose analytic bounds exceed the envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 icebergs), or whose largest $\lceil qty/peak \rceil$ exceeds 64, is never run. A draw is then discarded and re-drawn; a twin is recorded as a failed check. The slice count is re-checked from the tape and reported, as 0. After the run, the battery's I8 confirms the flags.

**Derived runs.** Each run below changes only what its row of the table says. Write $T_1, \dots, T_k$ for the touch orders in their drawn entry order, and $D_1$ for the best (least-worse) decoy.

| run | made from | change | on |
|---|---|---|---|
| `m1_reversal` | every valid ladder draw | the touch rows re-entered as $T_k, \dots, T_1$ in the same tape slots; the timestamp column keeps its values by slot, so it still grows down the tape | every draw |
| `self_match` | ladder draw | the aggressor's agent id set to the touch orders' agent id | first `SELF_MATCH_DRAWS` |
| `one_tick_over` | ladder draw | aggressor size $j \cdot q + 1$ tick | first `TWIN_DRAWS` |
| `one_tick_under` | ladder draw | aggressor size $j \cdot q - 1$ tick | first `TWIN_DRAWS` |
| `all_touch` | ladder draw | aggressor size $k \cdot q$ | first `TWIN_DRAWS` |
| `into_decoys` | ladder draw | aggressor size $k \cdot q + h$, $h = \max(1, \lfloor q/2 \rfloor)$ | first `TWIN_DRAWS` |
| `shared_ts` | ladder draw | every touch row's tape timestamp set to $T_1$'s value: the $k$ touch orders share one second | first `TWIN_DRAWS` |
| `shared_ts_reversed` | `shared_ts` | the touch rows re-entered in reverse order | first `TWIN_DRAWS` |
| `same_second` (r2) | ladder draw | every real row's tape timestamp set to the first real row's value: the whole tape, decoys and aggressor included, in one second | first `SAME_SECOND_DRAWS` |
| `same_second_reversed` (r2) | `same_second` | the touch rows re-entered as $T_k, \dots, T_1$ in the same tape slots | first `SAME_SECOND_DRAWS` |

**The derived-run precondition**, checked from the tape for every derived run (an aggregate assertion in section 7). It requires §2.3's bounds and the capacity margin, and the same touch and decoy order ids as the draw. The touch rows must enter in the draw's order, or in exactly its reverse on `m1_reversal`, `shared_ts_reversed` and `same_second_reversed`. Tape timestamps must never decrease down the tape. On `shared_ts*` the $k$ touch rows share one tape second and on `same_second*` every real row does. On every other derived run no two rows share a second. Below: the first seed of each construction in readable units beside the integers, a per-construction summary, the first ladder seed's derived tapes, and plots of the book shape and of the dataset's coverage.

In [ ]:
from collections import Counter

from omiecid_simulator.omiecid.encoding import decode_price, decode_qty

C_SIDE, C_PRICE, C_QTY, C_OID = synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY, synth.COL_OID
C_AGENT, C_TS, C_EXEC, C_CONTRACT, C_PEAK = (synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC,
                                             synth.COL_CONTRACT, synth.COL_PEAK)


def real_rows(tape):
    """The real rows (order_id >= 1), in tape (= submission) order."""
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def real_index(tape):
    """Tape row indices of the real rows, in order."""
    return [int(i) for i in np.nonzero(np.asarray(tape)[:, C_OID] >= 1)[0]]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def other(side):
    return SELL if int(side) == BUY else BUY


def better(rest_side, p, p_ref):
    """True when resting price p is strictly better for a taker than p_ref (§6.1)."""
    return int(p) < int(p_ref) if int(rest_side) == SELL else int(p) > int(p_ref)


def crosses(rest_side, p_rest, p_agg):
    """§6.1 / §6.2 item 2: sell price <= buy price."""
    return int(p_rest) <= int(p_agg) if int(rest_side) == SELL else int(p_rest) >= int(p_agg)


def max_slices(tape):
    """Largest ceil(qty / peak) over the tape's iceberg rows (0 when there are none)."""
    ice = [r for r in real_rows(tape) if int(r[C_EXEC]) == ICE]
    return max((-(-int(r[C_QTY]) // max(1, int(r[C_PEAK]))) for r in ice), default=0)


def shape(tape):
    """The tape read as 'resting rows in entry order, then one aggressor', with its touch and decoys.

    Returns None when the rows before the last are not all on the side opposite the last row."""
    rows = real_rows(tape)
    if len(rows) < 2:
        return None
    rest, agg = rows[:-1], rows[-1]
    rs = int(rest[0][C_SIDE])
    if any(int(r[C_SIDE]) != rs for r in rest) or int(agg[C_SIDE]) == rs:
        return None
    best = rest[0][C_PRICE]
    for r in rest:
        if better(rs, r[C_PRICE], best):
            best = r[C_PRICE]
    touch = [int(r[C_OID]) for r in rest if int(r[C_PRICE]) == int(best)]
    decoys = [int(r[C_OID]) for r in rest if int(r[C_PRICE]) != int(best)]
    pos = {int(r[C_OID]): i for i, r in enumerate(rest)}
    by_oid = {int(r[C_OID]): r for r in rows}
    return {"rest": rest, "agg": agg, "rest_side": rs, "touch": touch, "decoys": decoys, "pos": pos,
            "by_oid": by_oid, "p_touch": int(best)}


def bounds_ok(tape):
    return all(0 <= int(r[C_PRICE]) <= PRICE_INT_MAX and QTY_INT_MIN <= int(r[C_QTY]) <= QTY_INT_MAX
               for r in real_rows(tape))


def precondition_ladder(draw):
    """Every way a ladder draw departs from the dataset described above (empty list = as described)."""
    key, tape, p, bad = draw.construction, draw.tape, draw.params, []
    rows = real_rows(tape)
    sh = shape(tape)
    if sh is None:
        return ["not 'resting rows on one side, then one aggressor on the other'"]
    if any(int(r[C_EXEC]) != NON for r in rows):
        bad.append("a row is not NON")
    if len({int(r[C_CONTRACT]) for r in rows}) != 1:
        bad.append("more than one contract")
    if not bounds_ok(tape):
        bad.append("a row outside §2.3's bounds")
    if len({int(r[C_OID]) for r in rows}) != len(rows):
        bad.append("duplicate order ids")
    if not all(int(a[C_TS]) < int(b[C_TS]) for a, b in zip(rows[:-1], rows[1:])):
        bad.append("tape timestamps not strictly increasing down the tape")
    touch, decoys, agg, rs = sh["touch"], sh["decoys"], sh["agg"], sh["rest_side"]
    q = {int(sh["by_oid"][o][C_QTY]) for o in touch}
    k = len(touch)
    if k not in K_RANGE:
        bad.append(f"k = {k} touch orders (expected 2..6)")
    if len(q) != 1:
        bad.append(f"touch sizes differ: {sorted(q)}")
    if len(decoys) not in DECOY_RANGE:
        bad.append(f"{len(decoys)} decoys (expected 1..3)")
    if not crosses(rs, sh["p_touch"], agg[C_PRICE]):
        bad.append("the aggressor does not cross the touch")
    q_each = min(q) if q else 0
    q_agg = int(agg[C_QTY])
    j = q_agg // q_each if q_each else -1
    if not q_each or q_agg != j * q_each or not 1 <= j < k:
        bad.append(f"aggressor size {q_agg} is not j * q with 1 <= j < k (q = {q_each}, k = {k})")
    first_touch = min(sh["pos"][o] for o in touch)
    last_touch = max(sh["pos"][o] for o in touch)
    if key == LADDER and not all(sh["pos"][d] > last_touch for d in decoys):
        bad.append("a decoy entered before a touch order")
    if key == POT:
        if not all(sh["pos"][d] < first_touch for d in decoys):
            bad.append("a decoy entered after a touch order")
        if not all(crosses(rs, sh["by_oid"][d][C_PRICE], agg[C_PRICE]) for d in decoys):
            bad.append("a decoy does not cross the aggressor's limit")
    want = {"k": k, "j": j, "rest_side": rs, "agg_side": int(agg[C_SIDE]), "p_touch_int": sh["p_touch"],
            "q_each_int": q_each, "at_touch_oids": touch, "agg_oid": int(agg[C_OID]), "q_agg_int": q_agg,
            "p_agg_int": int(agg[C_PRICE])}
    want["decoy_oids" if key == LADDER else "early_decoy_oids"] = decoys
    want["n_decoys" if key == LADDER else "n_early"] = len(decoys)
    diff = {kk: (p.get(kk), v) for kk, v in want.items() if p.get(kk) != v}
    if diff:
        bad.append(f"params disagree with the tape (params, tape): {diff}")
    return bad


def precondition_pair(draw):
    """Every way a same-owner pair departs from the dataset described above."""
    rows, bad = real_rows(draw.tape), []
    cf = draw.aux_tapes.get("counterfactual")
    if len(rows) != 2:
        return [f"{len(rows)} real rows (expected 2)"]
    r, a = rows
    if int(r[C_EXEC]) != NON or int(a[C_EXEC]) != NON:
        bad.append("a row is not NON")
    if int(r[C_CONTRACT]) != int(a[C_CONTRACT]) or int(r[C_SIDE]) == int(a[C_SIDE]):
        bad.append("not opposite sides of one contract")
    if not crosses(r[C_SIDE], r[C_PRICE], a[C_PRICE]):
        bad.append("the pair does not cross")
    if not bounds_ok(draw.tape):
        bad.append("a row outside §2.3's bounds")
    if int(r[C_AGENT]) != int(a[C_AGENT]):
        bad.append("the two rows carry different agent ids")
    if int(draw.params.get("owner_agent_id", -1)) != int(r[C_AGENT]):
        bad.append("params owner_agent_id disagrees with the tape")
    if cf is None:
        bad.append("no counterfactual tape")
    else:
        cf = np.asarray(cf)
        t = np.asarray(draw.tape)
        if cf.shape != t.shape:
            bad.append("counterfactual shape differs")
        else:
            cols = [c for c in range(t.shape[1]) if c != C_AGENT]
            if not np.array_equal(cf[:, cols], t[:, cols]):
                bad.append("counterfactual differs outside the agent column")
            cr = real_rows(cf)
            if len(cr) != 2 or int(cr[0][C_AGENT]) == int(cr[1][C_AGENT]):
                bad.append("counterfactual agent ids are not distinct")
    return bad


def derived_tapes(draw):
    """The derived runs of section 4's table, for one ladder draw (tape copies; the draw is untouched)."""
    base = np.array(draw.tape, copy=True)
    idx = real_index(base)
    sh = shape(base)
    rows = {int(base[i, C_OID]): i for i in idx}
    t_slots = [rows[o] for o in sh["touch"]]          # tape slots of T_1..T_k, in entry order
    agg_i = idx[-1]
    q = int(base[t_slots[0], C_QTY])
    k = len(t_slots)
    out = {}

    def reverse_touch(t):
        t = t.copy()
        ts_col = t[:, C_TS].copy()
        t[t_slots] = t[t_slots[::-1]]
        t[:, C_TS] = ts_col                            # the timestamp column keeps its values by slot
        return t

    def with_agg_qty(qty):
        t = base.copy()
        t[agg_i, C_QTY] = qty
        return t

    out["m1_reversal"] = reverse_touch(base)
    sm = base.copy()
    sm[agg_i, C_AGENT] = int(base[t_slots[0], C_AGENT])
    out["self_match"] = sm
    out["one_tick_over"] = with_agg_qty(int(base[agg_i, C_QTY]) + 1)
    out["one_tick_under"] = with_agg_qty(int(base[agg_i, C_QTY]) - 1)
    out["all_touch"] = with_agg_qty(k * q)
    out["into_decoys"] = with_agg_qty(k * q + max(1, q // 2))
    shared = base.copy()
    shared[t_slots, C_TS] = int(base[t_slots[0], C_TS])
    out["shared_ts"] = shared
    shared_r = shared.copy()
    shared_r[t_slots] = shared[t_slots[::-1]]          # all touch timestamps are equal: nothing to keep by slot
    out["shared_ts_reversed"] = shared_r
    same = base.copy()
    same[idx, C_TS] = int(base[idx[0], C_TS])          # every real row in the first row's second
    out["same_second"] = same
    same_r = same.copy()
    same_r[t_slots] = same[t_slots[::-1]]              # one second throughout: nothing to keep by slot
    out["same_second_reversed"] = same_r
    return {kk: np.ascontiguousarray(v, dtype=np.int32) for kk, v in out.items()}


SIZE_TWINS = ("one_tick_over", "one_tick_under", "all_touch", "into_decoys")
TIE_TWINS = ("shared_ts", "shared_ts_reversed")            # r2: asserted exactly, like the size twins
EXACT_TWINS = SIZE_TWINS + TIE_TWINS
TWIN_VARIANTS = EXACT_TWINS
SAME_SECOND_RUNS = ("same_second", "same_second_reversed")
REVERSED_RUNS = ("m1_reversal", "shared_ts_reversed", "same_second_reversed")


def derived_precondition(draw, v, t):
    """Every way derived run `v` of `draw` departs from section 4's table (empty list = as described)."""
    sh0, sh = shape(draw.tape), shape(t)
    if sh is None:
        return ["not 'resting rows on one side, then one aggressor on the other'"]
    bad, rows = [], real_rows(t)
    ts = {int(r[C_OID]): int(r[C_TS]) for r in rows}
    if not all(int(a[C_TS]) <= int(b[C_TS]) for a, b in zip(rows[:-1], rows[1:])):
        bad.append("tape timestamps decrease down the tape")
    if sorted(sh["touch"]) != sorted(sh0["touch"]) or sorted(sh["decoys"]) != sorted(sh0["decoys"]):
        bad.append("touch or decoy order ids differ from the draw's")
    want = sh0["touch"][::-1] if v in REVERSED_RUNS else sh0["touch"]
    if sh["touch"] != want:
        bad.append(f"touch entry order {sh['touch']} (expected {want})")
    if v in TIE_TWINS:
        if len({ts[o] for o in sh["touch"]}) != 1:
            bad.append("the touch rows do not share one tape second")
    elif v in SAME_SECOND_RUNS:
        if len(set(ts.values())) != 1:
            bad.append("the real rows do not share one tape second")
    elif len(set(ts.values())) != len(ts):
        bad.append("two rows share a tape second")
    return bad
DRAWN, DERIVED = {}, {}


def prepare(key, seed):
    """Draw seed `seed` of construction `key` and derive its runs (initial seeds and re-draws)."""
    d = synth.CONSTRUCTIONS[key](seed)
    DRAWN[(key, seed)] = d
    if key in LADDER_KEYS:
        DERIVED[(key, seed)] = derived_tapes(d)
    return d


for key in CONSTRUCTIONS:
    for s in range(SEED0, SEED0 + N_DRAWS[key]):
        prepare(key, s)


def rows_table(tape, title):
    sh = shape(tape)
    role = {}
    if sh is not None:
        if len(sh["rest"]) == 1:
            role[int(sh["rest"][0][C_OID])] = "resting"
        else:
            role.update({o: f"touch T{n}" for n, o in enumerate(sh["touch"], start=1)})
        role.update({o: "decoy" for o in sh["decoys"]})
        role[int(sh["agg"][C_OID])] = "aggressor"
    lines = [f"**{title}**", "",
             "| row | role | oid | side | price (EUR/MWh) | qty (MW) | price_int | qty_int | agent | ts |",
             "|---|---|---|---|---|---|---|---|---|---|"]
    for n, r in enumerate(real_rows(tape), start=1):
        lines.append(
            f"| {n} | {role.get(int(r[C_OID]), '')} | {int(r[C_OID])} | {side_name(r[C_SIDE])} | "
            f"{decode_price(int(r[C_PRICE]), CONFIG):.2f} | {decode_qty(int(r[C_QTY]), CONFIG):.1f} | "
            f"{int(r[C_PRICE])} | {int(r[C_QTY])} | {int(r[C_AGENT])} | {int(r[C_TS])} |")
    return "\n".join(lines)


for key in CONSTRUCTIONS:
    d = DRAWN[(key, SEED0)]
    display(Markdown(rows_table(d.tape, f"{key}, seed {SEED0} (digest `{d.digest()[:16]}…`, tape {d.tape.shape}, "
                                        f"hop bound {d.hop_bound})")))
display(Markdown(rows_table(DRAWN[(PAIR, SEED0)].aux_tapes["counterfactual"],
                            f"{PAIR}, seed {SEED0}: the distinct-owner counterfactual")))

for key in LADDER_KEYS:
    seeds = range(SEED0, SEED0 + N_DRAWS[key])
    ps = [DRAWN[(key, s)].params for s in seeds]
    nd = "n_decoys" if key == LADDER else "n_early"
    print(f"{key}: {len(ps)} seeds; k {dict(sorted(Counter(p['k'] for p in ps).items()))}; "
          f"j = 1: {sum(p['j'] == 1 for p in ps)}, j = k - 1: {sum(p['j'] == p['k'] - 1 for p in ps)}; "
          f"decoys {dict(sorted(Counter(p[nd] for p in ps).items()))}; "
          f"resting SELL {sum(p['rest_side'] == SELL for p in ps)}, BUY {sum(p['rest_side'] == BUY for p in ps)}")
_pp = [DRAWN[(PAIR, s)].params for s in range(SEED0, SEED0 + M6_DRAWS)]
print(f"{PAIR}: {len(_pp)} seeds; aggressor BUY {sum(p['agg_side'] == BUY for p in _pp)}, "
      f"SELL {sum(p['agg_side'] == SELL for p in _pp)}; q_agg < q_rest {sum(p['q_agg_int'] < p['q_rest_int'] for p in _pp)}, "
      f"= {sum(p['q_agg_int'] == p['q_rest_int'] for p in _pp)}, > {sum(p['q_agg_int'] > p['q_rest_int'] for p in _pp)}")
print("largest analytic bounds over all seeds: "
      + ", ".join(f"{a} {max(d.capacity_bounds[a] for d in DRAWN.values())}" for a in synth.AXES)
      + f"; largest ceil(qty/peak) {max(max_slices(d.tape) for d in DRAWN.values())}")

for v, t in DERIVED[(LADDER, SEED0)].items():
    display(Markdown(rows_table(t, f"{LADDER}, seed {SEED0}, derived run `{v}`")))

In [ ]:
import matplotlib.pyplot as plt

ROLE_COLOUR = {"touch": "tab:blue", "decoy": "tab:grey", "aggressor": "tab:orange"}
fig, axes = plt.subplots(1, 4, figsize=(17, 3.4))
for ax, key in zip(axes[:2], LADDER_KEYS):
    d = DRAWN[(key, SEED0)]
    sh = shape(d.tape)
    for n, r in enumerate(real_rows(d.tape)):
        o = int(r[C_OID])
        role = "aggressor" if o == int(sh["agg"][C_OID]) else ("touch" if o in sh["touch"] else "decoy")
        sign = 1 if int(r[C_SIDE]) == BUY else -1
        ax.barh(decode_price(int(r[C_PRICE]), CONFIG), decode_qty(int(r[C_QTY]), CONFIG) * sign, height=0.04,
                left=0, color=ROLE_COLOUR[role], alpha=0.35 + 0.6 * (n + 1) / len(real_rows(d.tape)))
        ax.annotate(f"#{o}", (decode_qty(int(r[C_QTY]), CONFIG) * sign, decode_price(int(r[C_PRICE]), CONFIG)),
                    fontsize=6)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("quantity (MW): buy right, sell left")
    ax.set_ylabel("limit price (EUR/MWh)")
    ax.set_title(f"{key.split('.')[1]}, seed {SEED0}\n(blue touch, grey decoy, orange aggressor; #oid)", fontsize=8)

for key, mk in zip(LADDER_KEYS, ("o", "x")):
    ps = [DRAWN[(key, s)].params for s in range(SEED0, SEED0 + N_DRAWS[key])]
    kj = Counter((p["k"], p["j"]) for p in ps)
    axes[2].scatter([a for a, _ in kj], [b for _, b in kj], s=[4 * n for n in kj.values()], marker=mk,
                    alpha=0.6, label=key.split(".")[1])
axes[2].set_xlabel("k (touch orders)")
axes[2].set_ylabel("j (orders the aggressor takes)")
axes[2].set_title("(k, j) coverage; marker area = draws")
axes[2].legend(fontsize=7)

steps = [DRAWN[(POT, s)].params["step_ticks"] for s in range(SEED0, SEED0 + POT_DRAWS)]
axes[3].hist(steps, bins=np.arange(0.5, 51.5, 1.0), color="tab:grey")
axes[3].set_xlabel("decoy step s (ticks)")
axes[3].set_ylabel("draws")
axes[3].set_title("price_over_time: decoy spacing")
plt.tight_layout()
plt.show()

## 5. Known result

**Notation.** Integer ticks throughout. The tape's real rows, in submission order, are the resting rows $r_1, \dots, r_m$ (one side, $\sigma$) and then the aggressor $a$ (the other side), with limit $p_a$, size $q_a$ and order id $o_a$. Row $r_i$ has limit $p_i$, size $q_i$, order id $o_i$ and **entry index** $i$. Every row is NON on one contract (section 4's precondition).

**(R) Ranking** (§6.1, §5.2). Every $r_i$ meets no counterpart on entry and rests (§4.5). Each receives its timestamp on entry, from one global queue, unique and sequential (§5.2), so its timestamp order **is** its entry order $i$. That holds also when several rows share one tape second: a tape enters its rows in row order, and the queue still stamps them one after another (section 2). The resting side is ranked by
$$\kappa(r_i) = \begin{cases} (p_i,\ i) & \sigma = \text{SELL (lowest price first)}\\ (-p_i,\ i) & \sigma = \text{BUY (highest price first)} \end{cases}$$
in increasing lexicographic order.

**(W) The walk** (§6.2 items 1, 3, 6; §6.1's $\ge$). Start with remainder $\rho = q_a$. Take the resting rows in $\kappa$ order. While $\rho > 0$ and the next row crosses ($p_i \le p_a$ for a SELL row, $p_i \ge p_a$ for a BUY row), fill $f_i = \min(\rho, q_i)$ at **$p_i$** and set $\rho \leftarrow \rho - f_i$. Stop at the first row that does not cross: every later row is worse.

**What is asserted, per run:**
- **(S) filled set** $= \{o_i : f_i > 0\}$ (the card's pass criterion);
- **(Q) counterparty sequence** $=$ the list $[o_i,\ o_a,\ p_i,\ f_i,\ \text{on } a\text{'s submission}]$ in walk order. Every fill is written by $a$'s submission; no resting row's submission writes a trade.
- **(B) book afterwards** $=$ every $r_i$ with $q_i - f_i > 0$ at $(p_i,\ q_i - f_i)$, plus $a$ at $(p_a, \rho)$ when $\rho > 0$ (§4.5, NON). Nothing else rests.
- **(T) stored timestamps** (§5.2): in the book just before $a$'s submission, each $o_i$ holds exactly one entry, and the engine's stored timestamps $(\text{sec}, \text{nsec})$, listed in entry order $i = 1, \dots, m$, are **strictly increasing**. That makes them unique.
- **(T=) same-second stamps** (§5.2, §13.1; r2): a *same-second group* is a set of two or more resting rows that share one tape timestamp value. In every group, each row holds exactly one stored entry, and the stored stamps in entry order are strictly increasing, so they are unique and sequential. This is recorded twice so a failure says where it lies: once per run over the same-price parts of the groups (rows that share second **and** price), and once over the groups that hold **more than one price** (the queue is global). A run with no group of a kind records no assertion of that kind. (T) already implies (T=); (T=) names the second in which a failure lies.

**Closed form on a ladder draw** (from `draw.params`, cross-checked against (W) on every draw). The touch orders $T_1, \dots, T_k$, in entry order, each of size $q$ at $p_{touch}$, and $q_a = j \cdot q$ with $1 \le j < k$:
$$\text{filled} = \{T_1, \dots, T_j\},\qquad \text{sequence} = [(T_1, q), \dots, (T_j, q)] \text{ at } p_{touch},\qquad \text{rest} = \{T_{j+1}, \dots, T_k\} \cup \text{decoys, all at full size};$$
and the aggressor does not rest. The decoys are strictly worse than the touch and the aggressor is used up inside it, so they are never reached. In `price_over_time` that holds **even though they entered first and cross the aggressor's limit**.

**Derived runs, by construction** ($T_1..T_k$, $q$, $j$, $D_1$ as in section 4):

| run | filled sequence (each at its maker's limit) | rests afterwards, besides the untouched |
|---|---|---|
| draw | $T_1, \dots, T_j$, each $q$ | — |
| `m1_reversal` | $T_k, T_{k-1}, \dots, T_{k-j+1}$, each $q$ (entry order is now reversed) | — |
| `self_match` | identical to the draw (M6) | — |
| `one_tick_over` | $T_1, \dots, T_j$ each $q$, then $T_{j+1}$ for 1 tick | $T_{j+1}$ with $q - 1$ |
| `one_tick_under` | $T_1, \dots, T_{j-1}$ each $q$, then $T_j$ for $q - 1$ | $T_j$ with 1 tick |
| `all_touch` | $T_1, \dots, T_k$, each $q$ | — (every decoy untouched) |
| `into_decoys` | $T_1, \dots, T_k$ each $q$, then $D_1$ for $h$ at $D_1$'s limit | $D_1$ with $q - h$ |
| `shared_ts` | $T_1, \dots, T_j$, each $q$ (the $k$ touch orders share one second) | — |
| `shared_ts_reversed` | $T_k, \dots, T_{k-j+1}$, each $q$ | — |
| `same_second` | $T_1, \dots, T_j$, each $q$ (the whole tape in one second) | — |
| `same_second_reversed` | $T_k, \dots, T_{k-j+1}$, each $q$ | — |

**(M1) The reversed runs.** `m1_reversal`, `shared_ts_reversed` and `same_second_reversed` re-enter the touch orders in reverse order. Each is asserted exactly by (S), (Q), (B), (T) and (T=) like every other run. On top of that, the battery's relation M1 must hold between each one and its unreversed partner (the draw, `shared_ts`, `same_second`): equal aggregate volume, and no price-priority violation involving a touch order. In r1 the two shared-second twins were asserted only up to the tie order. r2 asserts them exactly, because §5.2 and §13.1 together fix the order inside a second (section 2). **The order inside a second is the row order of the tape**: the expectation never reads the timestamp column to rank.

**(M6) Ownership.** Same-owner pair: the walk (W) for its two rows gives one fill of $\min(q_{rest}, q_{agg})$ at $p_{rest}$. The trade list, as (price, quantity, passive id, aggressor id, trade sec, trade nsec), must **equal** the counterfactual's entry for entry, and its buyer and seller agent ids must be the counterfactual's mapped through the relabelling (`check_m6_ownership_blindness`). The same holds for each ladder draw against its `self_match` run.

The code below implements (W) as **one function of the tape rows**, used unchanged for every draw and every derived run. It does not read `draw.expected` and it does not run the engine. Three wrong rankers are written beside it, run on the same tapes, so section 7 can show the dataset tells each of them apart from (W): **time-only** (entry order, ignoring price), **latest-first** at a price, and **order-id** (lowest id first at a price). The order-id ranker is also what a book does when it breaks equal stored stamps by order id, so it is separated on all three reversed runs, the two same-second ones included.

In [ ]:
def walk(tape, key="price_time"):
    """(W): the fills one aggressor (the last real row) takes from the resting rows, in integer ticks.

    key="price_time" is (R), the rule. "time_only", "latest_first" and "oid" are the three wrong rankers of
    section 5, used only to show that the dataset separates them from the rule. Returns None if the tape is
    not 'resting rows on one side, then one aggressor'."""
    sh = shape(tape)
    if sh is None:
        return None
    rest, a, rs = sh["rest"], sh["agg"], sh["rest_side"]
    sgn = 1 if rs == SELL else -1

    def rank(i):
        p, o = sgn * int(rest[i][C_PRICE]), int(rest[i][C_OID])
        return {"price_time": (p, i), "time_only": (i,), "latest_first": (p, -i), "oid": (p, o)}[key]

    left = {int(r[C_OID]): int(r[C_QTY]) for r in rest}
    rho, fills = int(a[C_QTY]), []
    for i in sorted(range(len(rest)), key=rank):
        if rho == 0:
            break
        r = rest[i]
        if not crosses(rs, r[C_PRICE], a[C_PRICE]):
            if key == "price_time":
                break        # ranked best first: nothing after this row crosses either
            continue         # a wrong ranker may still reach a crossing row further down its order
        f = min(rho, left[int(r[C_OID])])
        fills.append([int(r[C_OID]), int(a[C_OID]), int(r[C_PRICE]), f, True])
        left[int(r[C_OID])] -= f
        rho -= f
    book = {"BUY": [], "SELL": []}
    for r in rest:
        if left[int(r[C_OID])] > 0:
            book[side_name(rs)].append([int(r[C_PRICE]), left[int(r[C_OID])], int(r[C_OID])])
    if rho > 0 and int(a[C_EXEC]) == NON:
        book[side_name(a[C_SIDE])].append([int(a[C_PRICE]), rho, int(a[C_OID])])
    return {"agg_oid": int(a[C_OID]), "agg_side": int(a[C_SIDE]), "rest_side": rs, "q_agg_int": int(a[C_QTY]),
            "fills": fills, "filled_set": sorted({f[0] for f in fills}),
            "book": {s: sorted(v) for s, v in book.items()},
            "entry_order": [int(r[C_OID]) for r in rest], "entry_ts": [int(r[C_TS]) for r in rest],
            "entry_price": [int(r[C_PRICE]) for r in rest], "touch": sh["touch"], "decoys": sh["decoys"],
            "p_touch": sh["p_touch"], "total_qty_int": int(a[C_QTY]) - rho}


def expectation(tape):
    """(S), (Q), (B) and the entry order (T) reads, from the tape rows only."""
    return walk(tape, "price_time")


def same_second_groups(e):
    """(T=): the same-second groups of the resting rows, each as order ids in entry order.

    Returns {"same_price": the parts of the groups that also share one price (two rows or more),
             "across_prices": the groups that hold more than one price}."""
    by_ts = {}
    for o, t in zip(e["entry_order"], e["entry_ts"]):
        by_ts.setdefault(t, []).append(o)
    price = dict(zip(e["entry_order"], e["entry_price"]))
    groups = [g for g in by_ts.values() if len(g) >= 2]
    same_px = []
    for g in groups:
        by_p = {}
        for o in g:
            by_p.setdefault(price[o], []).append(o)
        same_px += [h for h in by_p.values() if len(h) >= 2]
    return {"same_price": same_px, "across_prices": [g for g in groups if len({price[o] for o in g}) >= 2]}


def closed_form(draw):
    """Section 5's closed form for a ladder draw, from draw.params (decoy limits read from their tape rows)."""
    p = draw.params
    touch, j, q, pt, agg = p["at_touch_oids"], p["j"], p["q_each_int"], p["p_touch_int"], p["agg_oid"]
    decoys = p["decoy_oids"] if draw.construction == LADDER else p["early_decoy_oids"]
    by_oid = {int(r[C_OID]): r for r in real_rows(draw.tape)}
    rest = [[pt, q, o] for o in touch[j:]] + [[int(by_oid[o][C_PRICE]), q, o] for o in decoys]
    book = {"BUY": [], "SELL": []}
    book[side_name(p["rest_side"])] = sorted(rest)
    return {"fills": [[o, agg, pt, q, True] for o in touch[:j]], "filled_set": sorted(touch[:j]), "book": book}


def derived_table(draw, variant):
    """Section 5's table of derived runs, written out row by row as [passive oid, price_int, qty_int]
    (checked against `expectation` on the first seed of each ladder construction)."""
    sh = shape(draw.tape)
    T, D, q = sh["touch"], sh["decoys"], int(sh["by_oid"][sh["touch"][0]][C_QTY])
    j, k, pt = int(sh["agg"][C_QTY]) // q, len(T), sh["p_touch"]
    sgn = 1 if sh["rest_side"] == SELL else -1
    d1 = min(D, key=lambda o: sgn * int(sh["by_oid"][o][C_PRICE]))      # the best (least-worse) decoy
    h = max(1, q // 2)
    rows = {
        "draw": [[o, pt, q] for o in T[:j]],
        "m1_reversal": [[o, pt, q] for o in T[::-1][:j]],
        "self_match": [[o, pt, q] for o in T[:j]],
        "one_tick_over": [[o, pt, q] for o in T[:j]] + [[T[j], pt, 1]],
        "one_tick_under": [[o, pt, q] for o in T[:j - 1]] + [[T[j - 1], pt, q - 1]],
        "all_touch": [[o, pt, q] for o in T],
        "into_decoys": [[o, pt, q] for o in T] + [[d1, int(sh["by_oid"][d1][C_PRICE]), h]],
        "shared_ts": [[o, pt, q] for o in T[:j]],
        "shared_ts_reversed": [[o, pt, q] for o in T[::-1][:j]],
        "same_second": [[o, pt, q] for o in T[:j]],
        "same_second_reversed": [[o, pt, q] for o in T[::-1][:j]],
    }
    return rows[variant]


EXPECTED = {ks: expectation(d.tape) for ks, d in DRAWN.items()}

for key in CONSTRUCTIONS:
    for s in range(SEED0, SEED0 + 3):
        e = EXPECTED[(key, s)]
        print(f"{key} seed {s}: fills {[(f[0], f[2], f[3]) for f in e['fills']]} (oid, price_int, qty_int) "
              f"-> filled set {e['filled_set']}; rests {e['book']}")

TABLE_AGREES = True
for key in LADDER_KEYS:
    d = DRAWN[(key, SEED0)]
    tapes = {"draw": d.tape, **DERIVED[(key, SEED0)]}
    for v in ("draw", "m1_reversal", "self_match") + EXACT_TWINS + SAME_SECOND_RUNS:
        got = [[f[0], f[2], f[3]] for f in expectation(tapes[v])["fills"]]
        want = derived_table(d, v)
        TABLE_AGREES &= got == want
        print(f"{key} seed {SEED0} {v:<15}: expectation {got}  table {want}  {'agree' if got == want else 'DISAGREE'}")

print()
for key in LADDER_KEYS:
    d = DRAWN[(key, SEED0)]
    print(f"{key} seed {SEED0}, the wrong rankers on the draw: "
          + "; ".join(f"{k_} -> {walk(d.tape, k_)['filled_set']}" for k_ in ("price_time", "time_only", "latest_first", "oid"))
          + f"; on m1_reversal: oid -> {walk(DERIVED[(key, SEED0)]['m1_reversal'], 'oid')['filled_set']}, "
          f"rule -> {expectation(DERIVED[(key, SEED0)]['m1_reversal'])['filled_set']}")

print()
for key in LADDER_KEYS:
    for v in TIE_TWINS + SAME_SECOND_RUNS:
        e_ = expectation(DERIVED[(key, SEED0)][v])
        print(f"{key} seed {SEED0} {v:<20}: same-second groups {same_second_groups(e_)}; "
              f"rule -> {e_['filled_set']}, oid -> {walk(DERIVED[(key, SEED0)][v], 'oid')['filled_set']}")

## 6. Run

### Eager and jitted (wired at K3)

> **K3: this section is yours; sections 1–5 and 7 are locked (independence rule 2).** A disagreement with the expectation is escalated as a `SCOPE-CHANGE` candidate with both readings, never resolved by editing sections 5 or 7.

Section 7 consumes exactly **one name** from this section. Until it is callable, section 7 records a failing `section 6 wired` check and the notebook ends `NOT CERTIFIED`.

**`run_tape(tape, *, jit) -> RunRecord`**: the battery's record builder on one tape, i.e. `tests.omiecid.reference.omie_invariants.build_record(tape, jit=jit)`, on one fresh book of an OPEN contract with the default configs (`OMIEProductConfig()`, `OMIEMarketConfig()`), no scheduled cancellations, no gate-closure expiry, and the default capacity bounds (`tape_capacity_bounds(tape)`). `jit=False` runs eager (the builder wraps the run in `jax.disable_jit()`); `jit=True` runs jitted. The tape is passed **as is**: a generator draw's `DRAWN[(key, seed)].tape`, a pair's `DRAWN[(PAIR, seed)].aux_tapes["counterfactual"]`, or one of the derived runs `DERIVED[(key, seed)][variant]` (section 4). Nothing in the tape may be rewritten, and that includes its timestamp column.

Section 7 reads these `RunRecord` fields: `fills` (each `Fill`'s `price_int`, `qty_int`, `passive_oid`, `aggressor_oid`, `submission`, `row_index`, `sec`, `nsec`, `buyer_agent`, `seller_agent`), `accepted` (its keys and each `AcceptedOrder.submission`), `pre_snapshots[s].resting(side)` (the order id and the stored `sec`, `nsec` of each resting row just before the aggressor's submission $s$), `final_snapshot.resting(side)`, `observed_capacity`, and the whole record through `run_battery`, `compare_records`, `check_m1_tie_permutation` and `check_m6_ownership_blindness`.

Section 7 calls `run_tape(..., jit=True)` on every draw and every derived run it uses, and `run_tape(..., jit=False)` on the eager selection of section 3. A seed beyond the initial range (a re-draw after a discard) is prepared with `prepare(key, seed)` before the call.

In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_tape(tape, *, jit) -> RunRecord, per the contract above.
#
# The battery's record builder on one fresh book of one OPEN contract, with the default
# configs, no scheduled cancellations, no gate-closure expiry and the default analytic
# capacity bounds (tape_capacity_bounds(tape)). jit=False wraps the whole run in
# jax.disable_jit() (inside build_record); jit=True runs it jitted. The tape is passed as
# is: no column, including the timestamp column, is rewritten.
# ---------------------------------------------------------------------------------------
from omiecid_simulator.omiecid.contract import ContractStatus
from omiecid_simulator.omiecid.omie_config import OMIEMarketConfig
from tests.omiecid.reference.omie_invariants import build_record

MARKET = OMIEMarketConfig()


def run_tape(tape, *, jit):
    """One run of `tape` (a draw's tape, a pair's counterfactual or a derived run, unmodified) on a fresh
    OPEN book."""
    return build_record(tape, product_config=CONFIG, market_config=MARKET,
                        contract_status=ContractStatus.OPEN, cancels_before={},
                        expire_at_gct=False, capacity_bounds=None, jit=jit)


print(f"run_tape: build_record on one OPEN contract, {type(CONFIG).__name__}() + {type(MARKET).__name__}(), "
      f"no cancels, no GCT expiry, default capacity bounds")


## 7. Certification

The oracle comparison by order id and in integer ticks comes first, then the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per valid ladder draw** (so an auditor can check the exact integers from the results JSON alone, not only a folded flag), each with its expected and observed value:
1. `<key> seed s: filled order ids` — (S), the sorted filled set;
2. `<key> seed s: counterparty sequence` — (Q), the list `[passive oid, aggressor oid, price_int, qty_int, on the aggressor's submission]` in trade-row order;
3. `<key> seed s: book after` — (B), the resting `[price_int, qty_int, oid]` per side;
4. `<key> seed s: stored timestamps` — (T), the observed `[oid, [[sec, nsec], ...]]` per resting order in entry order, passing when each order holds exactly one entry and the stamps strictly increase;
5. `… same-second stamps unique and sequential in entry order (same price | across prices)` — (T=), on every run that has a same-second group of that kind (r2), with the groups as expected value and each group's observed `[oid, [[sec, nsec], ...]]` as observed value.

The **M1 reversal** of every ladder draw records the same four, named `… m1_reversal: …`, plus `… m1_reversal: M1 relation` (the battery's `check_m1_tie_permutation` between the draw and its reversal, with the touch orders as the tie set) and one assertion that its battery passed. The **self-match** runs record (S), (Q), the battery, and `… self_match: M6 relation` (`check_m6_ownership_blindness`, aggressor's agent mapped to the makers'). The **twins** (size and shared-second alike) record (S), (Q), (B), (T), (T=) and the battery, and `shared_ts_reversed` adds the M1 relation against `shared_ts`. The **same-second runs** (r2) record (S), (Q), (B), (T), (T=) and the battery, and `same_second_reversed` adds the M1 relation against `same_second`.

**Per same-owner pair:** (S), (Q), (B), (T) on the same-owner run; `… counterfactual: battery passes`; `… trade list equals the counterfactual` (exact list of `[price_int, qty_int, passive oid, aggressor oid, sec, nsec]`); and `… M6 relation` (`check_m6_ownership_blindness`, agents mapped from the counterfactual's to the same owner).

**Aggregate assertions:** the section-6 wiring; the dataset precondition per construction (section 4), listing failing seeds; the closed form of section 5 against (W) on every ladder draw; section 5's table of derived runs against (W); every derived run as section 4's derived-run precondition describes (bounds, margin, entry order, shared seconds); **separation** — the time-only ranker disagrees with (W) on every `price_over_time` draw, the latest-first ranker on every ladder draw, and the order-id ranker on every `m1_reversal`, `shared_ts_reversed` and `same_second_reversed` run; coverage — every $k \in [2,6]$, decoy count 1–3, $j = 1$ and $j = k - 1$ (for some $k \ge 3$), and both resting sides, per ladder construction; every $k \in [2,6]$ and both resting sides among the same-second runs, per ladder construction; and the four aggressor-side × size-order classes of the pair; and the eager comparisons, one per required class and derived-run variant, a missing one recorded as a failure.

The battery runs on every draw's jitted record with `expects_trade=True` (every C03 tape crosses). A draw the battery marks I8 `invalid` is discarded and replaced by the next seed. For derived runs the battery is folded into one assertion each, since a derived run is not a draw. A battery or relation status of `not_applicable` counts as satisfied (carried-forward judging rule), and every list an assertion records is recorded in full. A per-run table (all failing rows, then the first 20 passing ones) is shown before the full certificate table.

In [ ]:
import traceback
from collections import defaultdict

from tests.omiecid.reference.omie_invariants import (
    check_m1_tie_permutation, check_m6_ownership_blindness, compare_records, run_battery,
)

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Price-time priority, ties, and ownership blindness",
    known_result="the filled set is exactly the j earliest touch order ids, filled in entry order at the touch "
                 "price (exact counterparty sequence); decoys and the other k - j touch orders untouched; the "
                 "stored timestamps of the resting orders unique and strictly increasing in entry order, also "
                 "among orders that enter in the same second, at one price and across prices (r2); the "
                 "same-owner pair trades, with a trade list identical to the distinct-owner counterfactual (M6); "
                 "an entry-order reversal moves the filled set with it (M1)",
    oracle_kind="exact-parametric, plus M1 and M6", quotes=QUOTES, citations=CITATIONS,
    constructions=CONSTRUCTIONS, requested_draws=REQUESTED, runtime_forecast_s=180.0, started_at=T0,
    downstream="C04 (survivors keep this queue order after a cancellation), C05 (the interloper fills after "
               "the residual), C06 (levels consumed in rank order), C08-C09 (slices queue by timestamp), C15 (M6 "
               "at scale); every passive-fill figure of Track B (queue position decides who is filled)",
)

R_RANK = "§6.1; §11.1 (price-time priority)"
R_SEQ = "§6.1; §6.2 items 1, 3 (rank order, maker price)"
R_BOOK = "§4.5 (NON: book entry if unmatched); §6.2 item 1"
R_TS = "§5.2 (single global queue: unique, sequential)"
R_TS_SAME = "§5.2 (single global queue: unique, sequential); §13.1 (whole-second submission times)"
R_M1 = "§6.1 tie-break; relation M1"
R_M6 = "§6.6 (no self-trade prevention); relation M6"
R_DATA = "dataset precondition (section 4); §2.3"


def guarded(label, seed, fn, *args, **kw):
    """Call a section-6 function; a raise is recorded as a failed check, never propagated."""
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, expected="a RunRecord", observed=type(exc).__name__,
                   detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1], rule=R_RANK)
        return None


def observe(rec, e):
    """What the engine did, read from the record, in the expectation's terms."""
    agg = e["agg_oid"]
    a_sub = int(rec.accepted[agg].submission) if agg in rec.accepted else None
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    seq = [[int(f.passive_oid), int(f.aggressor_oid), int(f.price_int), int(f.qty_int),
            a_sub is not None and int(f.submission) == a_sub] for f in fills]
    book = {side_name(sd): sorted([int(x[0]), int(x[1]), int(x[2])] for x in rec.final_snapshot.resting(sd))
            for sd in (BUY, SELL)}
    ts = None
    if a_sub is not None and 0 <= a_sub < len(rec.pre_snapshots):
        stamps = defaultdict(list)
        for x in rec.pre_snapshots[a_sub].resting(e["rest_side"]):
            stamps[int(x[2])].append([int(x[4]), int(x[5])])
        ts = [[o, stamps.get(o, [])] for o in e["entry_order"]]
    return {"seq": seq, "filled_set": sorted({s_[0] for s_ in seq}), "book": book, "ts": ts}


def ts_ok(ts):
    """(T): one stored entry per resting order, stamps strictly increasing in entry order."""
    if ts is None or any(len(st) != 1 for _, st in ts):
        return False
    stamps = [tuple(st[0]) for _, st in ts]
    return all(a < b for a, b in zip(stamps[:-1], stamps[1:]))


def group_ts_ok(ts, groups):
    """(T=): in every group, one stored entry per order and stamps strictly increasing in entry order."""
    if ts is None:
        return False
    st = {o: s for o, s in ts}
    for g in groups:
        if any(len(st.get(o, [])) != 1 for o in g):
            return False
        stamps = [tuple(st[o][0]) for o in g]
        if not all(a < b for a, b in zip(stamps[:-1], stamps[1:])):
            return False
    return True


def group_obs(ts, groups):
    st = dict(ts) if ts is not None else {}
    return [[[o, st.get(o)] for o in g] for g in groups]


def check_exact(label, seed, rec, e, *, parts=("S", "Q", "B", "T")):
    """The per-run assertions (S), (Q), (B), (T) and (T=) of section 5. Returns (observation, all passed)."""
    o = observe(rec, e)
    ok = True
    if "S" in parts:
        ok &= cert.check_equal(f"{label}: filled order ids", o["filled_set"], e["filled_set"], seed=seed,
                               unit="other", rule=R_RANK)
    if "Q" in parts:
        ok &= cert.check_equal(f"{label}: counterparty sequence [passive, aggressor, price_int, qty_int, "
                               f"on aggressor's submission]", o["seq"], e["fills"], seed=seed, unit="other",
                               rule=R_SEQ)
    if "B" in parts:
        ok &= cert.check_equal(f"{label}: book after [price_int, qty_int, oid]", o["book"], e["book"], seed=seed,
                               unit="other", rule=R_BOOK)
    if "T" in parts:
        ok &= cert.check(f"{label}: stored timestamps unique and increasing in entry order", ts_ok(o["ts"]),
                         seed=seed, expected={"entry_order": e["entry_order"],
                                              "stamps": "one [sec, nsec] each, strictly increasing"},
                         observed=o["ts"], unit="other", rule=R_TS)
        groups = same_second_groups(e)
        for kind, gs in (("same price", groups["same_price"]), ("across prices", groups["across_prices"])):
            if gs:
                ok &= cert.check(f"{label}: same-second stamps unique and sequential in entry order ({kind})",
                                 group_ts_ok(o["ts"], gs), seed=seed,
                                 expected={"groups (oids in entry order)": gs,
                                           "stamps": "one [sec, nsec] each, strictly increasing in each group"},
                                 observed=group_obs(o["ts"], gs), unit="other", rule=R_TS_SAME)
    return o, bool(ok)


def battery_check(label, seed, rec, rule):
    rep = run_battery(rec, expects_trade=True)
    return cert.check(f"{label}: battery (RB, I1-I9) passes", rep.passed and rep.valid, seed=seed,
                      expected="pass / not_applicable", observed=rep.statuses(), unit="other", rule=rule)


def relation_check(label, seed, result, rule):
    # carried-forward judging rule: not_applicable is vacuously satisfied; the violation list is recorded in full
    return cert.check(f"{label}", result.status in ("pass", "not_applicable"), seed=seed,
                      expected="pass / not_applicable",
                      observed={"status": result.status, "evidence": dict(result.evidence),
                                "violations": [v.message for v in result.violations]},
                      unit="other", rule=rule)


def eager_compare(label, seed, tape, rec):
    """Run `tape` eager and compare with its jitted record; recorded as one eager_jit assertion."""
    rec_e = guarded(f"{label} (eager)", seed, run_tape, tape, jit=False)
    if rec is None or rec_e is None:
        diff = ["jitted run missing" if rec is None else "eager run missing"]
    else:
        diff = list(compare_records(rec_e, rec))
    cert.set_eager_jit(seed, diff == [], detail=f"{label}: " + (", ".join(diff) if diff else "identical"))
    return diff == []


def margin_breaches(tape, bounds):
    """The capacity-validity precondition, checked from the tape before it is run."""
    lim = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
           "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
    bad = [f"{k} bound {bounds.get(k)} > {m}" for k, m in lim.items() if int(bounds.get(k, 0)) > m]
    if max_slices(tape) > SLICE_MARGIN:
        bad.append(f"ceil(qty/peak) {max_slices(tape)} > {SLICE_MARGIN}")
    return bad


def trade_key(rec):
    return [[int(f.price_int), int(f.qty_int), int(f.passive_oid), int(f.aggressor_oid), int(f.sec), int(f.nsec)]
            for f in sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))]


def agent_map(base_tape, new_tape):
    """{agent id in base: agent id in new}, read per order id from the two tapes."""
    new = {int(r[C_OID]): int(r[C_AGENT]) for r in real_rows(new_tape)}
    return {int(r[C_AGENT]): new[int(r[C_OID])] for r in real_rows(base_tape) if int(r[C_OID]) in new}


# ---- section 5's own table agrees with the walk ---------------------------------------------------
cert.check("section 5: table of derived runs agrees with the walk (W) (first seed of each ladder construction)",
           TABLE_AGREES, expected=True, observed=TABLE_AGREES, unit="bool", rule=R_RANK)

wired = callable(run_tape)
cert.check("section 6 wired (run_tape defined)", wired, expected=True, observed=wired, unit="bool",
           detail="" if wired else "section 6 is still the K2 stub", rule=R_RANK)

PER_RUN, DATA_BAD, CLOSED_BAD, SEP_BAD = [], defaultdict(dict), defaultdict(list), defaultdict(list)
DERIVED_BAD, CLASS_COUNT, COVER, COVER_SS = [], Counter(), defaultdict(Counter), defaultdict(Counter)
EAGER_DONE = {}
PEAK = {"hop_bound": 0, "max_slices": 0}
REQUIRED_EAGER = []


def eager_once(tag, label, seed, tape, rec):
    """Force one eager comparison per tag (class or variant); later calls with the same tag do nothing."""
    if tag not in EAGER_DONE:
        EAGER_DONE[tag] = seed
        eager_compare(label, seed, tape, rec)


def run_derived(key, seed, v, label):
    """One jitted run of a derived tape, after its own capacity-margin check. Returns (tape, e, record)."""
    t = DERIVED[(key, seed)][v]
    e = expectation(t)
    bad = margin_breaches(t, synth.tape_capacity_bounds(t))
    if bad or not bounds_ok(t) or e is None:
        DERIVED_BAD.append(f"{label}: " + ("; ".join(bad) if bad else "outside §2.3 / not a ladder tape"))
        return t, e, None
    shape_bad = derived_precondition(DRAWN[(key, seed)], v, t)
    if shape_bad:
        DERIVED_BAD.append(f"{label}: " + "; ".join(shape_bad))   # recorded; the run still goes ahead
    if v in REVERSED_RUNS and walk(t, "oid")["filled_set"] == e["filled_set"]:
        SEP_BAD[f"{key}: oid on {v}"].append(seed)
    rec = guarded(f"{label} (jit)", seed, run_tape, t, jit=True)
    if rec is not None:
        battery_check(label, seed, rec, R_RANK)
    return t, e, rec


def run_ladder(key):
    valid, seed = 0, SEED0
    n, nd = N_DRAWS[key], ("n_decoys" if key == LADDER else "n_early")
    while valid < n and seed < SEED0 + 2 * n:
        draw = DRAWN.get((key, seed)) or prepare(key, seed)
        margin = margin_breaches(draw.tape, dict(draw.capacity_bounds))
        if margin:
            cert.discard(seed, f"{key}: pre-run capacity margin: " + "; ".join(margin))
            seed += 1
            continue
        e = EXPECTED.get((key, seed)) or expectation(draw.tape)
        EXPECTED[(key, seed)] = e
        rec = guarded(f"{key} seed {seed} (jit)", seed, run_tape, draw.tape, jit=True)
        if rec is not None and not cert.add_battery(run_battery(rec, expects_trade=True), seed=seed):
            seed += 1   # I8 invalid: discarded, re-drawn with the next seed
            continue
        valid += 1
        cert.record_case(seed, draw.tape, draw.params)
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        PEAK["max_slices"] = max(PEAK["max_slices"], max_slices(draw.tape))
        bad = precondition_ladder(draw)
        if bad:
            DATA_BAD[key][seed] = bad
        p = draw.params
        COVER[key]["k", p.get("k")] += 1
        COVER[key]["decoys", p.get(nd)] += 1
        COVER[key]["j=1"] += int(p.get("j") == 1)
        COVER[key]["j=k-1, k>=3"] += int(p.get("k", 0) >= 3 and p.get("j") == p.get("k", 0) - 1)
        cls = (key, side_name(p.get("rest_side", -1)))
        CLASS_COUNT[cls] += 1
        if e is None:
            cert.check(f"{key} seed {seed}: expectation defined", False, seed=seed, expected="a ladder tape",
                       observed=len(real_rows(draw.tape)), unit="count", rule=R_DATA)
            seed += 1
            continue
        if e["fills"] != closed_form(draw)["fills"] or e["book"] != closed_form(draw)["book"]:
            CLOSED_BAD[key].append(seed)
        sep = "time_only" if key == POT else "latest_first"
        if walk(draw.tape, sep)["filled_set"] == e["filled_set"]:
            SEP_BAD[f"{key}: {sep}"].append(seed)
        if rec is None:
            seed += 1
            continue
        cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
        lbl = f"{key} seed {seed}"
        o, ok = check_exact(lbl, seed, rec, e)
        PER_RUN.append({"key": key, "seed": seed, "run": "draw", "e": e, "o": o, "ok": ok})
        if valid <= EAGER_DRAWS:
            eager_compare(lbl, seed, draw.tape, rec)
            EAGER_DONE.setdefault(cls, seed)
        else:
            eager_once(cls, lbl, seed, draw.tape, rec)

        # ---- M1 reversal: every valid draw --------------------------------------------------------
        v = "m1_reversal"
        t, er, rr = run_derived(key, seed, v, f"{lbl} {v}")
        if rr is not None:
            o_r, ok_r = check_exact(f"{lbl} {v}", seed, rr, er)
            relation_check(f"{lbl} {v}: M1 relation (aggregate volume; no price-priority violation on a touch "
                           f"order)", seed, check_m1_tie_permutation(rec, rr, tie_oids=e["touch"]), R_M1)
            PER_RUN.append({"key": key, "seed": seed, "run": v, "e": er, "o": o_r, "ok": ok_r})
            eager_once((key, v), f"{lbl} {v}", seed, t, rr)

        # ---- M6 self-match: the first SELF_MATCH_DRAWS valid draws -------------------------------
        if valid <= SELF_MATCH_DRAWS:
            v = "self_match"
            t, es, rs_ = run_derived(key, seed, v, f"{lbl} {v}")
            if rs_ is not None:
                o_s, ok_s = check_exact(f"{lbl} {v}", seed, rs_, es, parts=("S", "Q"))
                relation_check(f"{lbl} {v}: M6 relation (trade list unchanged, agents mapped)", seed,
                               check_m6_ownership_blindness(rec, rs_, agent_map=agent_map(draw.tape, t)), R_M6)
                PER_RUN.append({"key": key, "seed": seed, "run": v, "e": es, "o": o_s, "ok": ok_s})
                eager_once((key, v), f"{lbl} {v}", seed, t, rs_)

        # ---- exact runs in pairs (unreversed, reversed) plus their M1 relation ----------------------
        def run_exact_group(variants, m1_pair=None):
            recs = {}
            for v in variants:
                t, et, rt = run_derived(key, seed, v, f"{lbl} {v}")
                if rt is None:
                    continue
                recs[v] = rt
                o_t, ok_t = check_exact(f"{lbl} {v}", seed, rt, et)
                PER_RUN.append({"key": key, "seed": seed, "run": v, "e": et, "o": o_t, "ok": ok_t})
                eager_once((key, v), f"{lbl} {v}", seed, t, rt)
            if m1_pair and all(v in recs for v in m1_pair):
                a_, b_ = m1_pair
                relation_check(f"{lbl} {b_}: M1 relation against {a_}", seed,
                               check_m1_tie_permutation(recs[a_], recs[b_], tie_oids=e["touch"]), R_M1)

        # ---- the boundary twins: the first TWIN_DRAWS valid draws --------------------------------
        if valid <= TWIN_DRAWS:
            run_exact_group(SIZE_TWINS)
            run_exact_group(TIE_TWINS, m1_pair=TIE_TWINS)

        # ---- the same-second runs (r2): the first SAME_SECOND_DRAWS valid draws --------------------
        if valid <= SAME_SECOND_DRAWS:
            COVER_SS[key]["k", p.get("k")] += 1
            COVER_SS[key]["resting", side_name(p.get("rest_side", -1))] += 1
            run_exact_group(SAME_SECOND_RUNS, m1_pair=SAME_SECOND_RUNS)
        seed += 1
    return valid


def run_pairs():
    valid, seed, n = 0, SEED0, N_DRAWS[PAIR]
    while valid < n and seed < SEED0 + 2 * n:
        draw = DRAWN.get((PAIR, seed)) or prepare(PAIR, seed)
        cf = draw.aux_tapes["counterfactual"]
        margin = (margin_breaches(draw.tape, dict(draw.capacity_bounds))
                  + margin_breaches(cf, synth.tape_capacity_bounds(cf)))
        if margin:
            cert.discard(seed, f"{PAIR}: pre-run capacity margin: " + "; ".join(margin))
            seed += 1
            continue
        e = EXPECTED.get((PAIR, seed)) or expectation(draw.tape)
        EXPECTED[(PAIR, seed)] = e
        rec = guarded(f"{PAIR} seed {seed} (jit)", seed, run_tape, draw.tape, jit=True)
        if rec is not None and not cert.add_battery(run_battery(rec, expects_trade=True), seed=seed):
            seed += 1
            continue
        valid += 1
        cert.record_case(seed, draw.tape, draw.params)
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        bad = precondition_pair(draw)
        if bad:
            DATA_BAD[PAIR][seed] = bad
        p = draw.params
        d_ = p.get("q_agg_int", 0) - p.get("q_rest_int", 0)
        cls = (PAIR, side_name(p.get("agg_side", -1)), "q_agg<q_rest" if d_ < 0 else "q_agg>q_rest" if d_ > 0 else "q_agg=q_rest")
        CLASS_COUNT[cls] += 1
        if rec is None or e is None:
            if e is None:
                cert.check(f"{PAIR} seed {seed}: expectation defined", False, seed=seed, expected="a two-row tape",
                           observed=len(real_rows(draw.tape)), unit="count", rule=R_DATA)
            seed += 1
            continue
        cert.record_capacity({**rec.observed_capacity, "hops": None})
        lbl = f"{PAIR} seed {seed}"
        o, ok = check_exact(lbl, seed, rec, e)
        rc = guarded(f"{lbl} counterfactual (jit)", seed, run_tape, cf, jit=True)
        if rc is not None:
            battery_check(f"{lbl} counterfactual", seed, rc, R_M6)
            ok &= cert.check_equal(f"{lbl}: trade list equals the counterfactual [price_int, qty_int, passive, "
                                   f"aggressor, sec, nsec]", trade_key(rec), trade_key(rc), seed=seed,
                                   unit="other", rule=R_M6)
            relation_check(f"{lbl}: M6 relation (counterfactual -> same owner)", seed,
                           check_m6_ownership_blindness(rc, rec, agent_map=agent_map(cf, draw.tape)), R_M6)
        PER_RUN.append({"key": PAIR, "seed": seed, "run": "draw", "e": e, "o": o, "ok": ok})
        if valid <= EAGER_DRAWS or cls not in EAGER_DONE:
            eager_compare(lbl, seed, draw.tape, rec)
            eager_compare(f"{lbl} counterfactual", seed, cf, rc)
            EAGER_DONE.setdefault(cls, seed)
        seed += 1
    return valid


VALID = {}
if wired:
    for key in LADDER_KEYS:
        VALID[key] = run_ladder(key)
    VALID[PAIR] = run_pairs()

# ---- aggregate assertions ------------------------------------------------------------------------
for key in CONSTRUCTIONS:
    bad = DATA_BAD.get(key, {})
    cert.check(f"dataset: every valid {key} draw is as section 4 describes (precondition)", wired and not bad,
               expected=[], observed={str(k): v for k, v in bad.items()}, unit="other",
               detail=(f"{len(bad)} draws depart: {list(bad)}" if bad
                       else f"{VALID.get(key, 0)}/{VALID.get(key, 0)} draws"), rule=R_DATA)
for key in LADDER_KEYS:
    cert.check(f"section 5: closed form agrees with the walk (W) on every {key} draw", wired and not CLOSED_BAD[key],
               expected=[], observed=list(CLOSED_BAD[key]), unit="other", rule=R_RANK)
_sep_names = [f"{POT}: time_only", f"{LADDER}: latest_first"] + [f"{key}: oid on {v}" for key in LADDER_KEYS
                                                                for v in REVERSED_RUNS]
for name in _sep_names:
    cert.check(f"dataset separates the rule from a wrong ranker on every run ({name})", wired and not SEP_BAD[name],
               expected=[], observed=list(SEP_BAD[name]), unit="other",
               detail=f"{len(SEP_BAD[name])} runs where the wrong ranker agrees with (W)" if SEP_BAD[name] else "",
               rule=R_RANK)
cert.check("dataset: every derived run is as section 4 describes (bounds, margin, entry order, shared seconds)",
           wired and not DERIVED_BAD, expected=[], observed=list(DERIVED_BAD), unit="other",
           rule="§2.3; capacity-validity precondition; section 4's derived-run precondition")
for key in LADDER_KEYS:
    want = {f"k={k}": ">= 1" for k in K_RANGE}
    want.update({f"decoys={d}": ">= 1" for d in DECOY_RANGE})
    want.update({"j=1": ">= 1", "j=k-1, k>=3": ">= 1", "resting BUY": ">= 1", "resting SELL": ">= 1"})
    c = COVER[key]
    got = {f"k={k}": c["k", k] for k in K_RANGE}
    got.update({f"decoys={d}": c["decoys", d] for d in DECOY_RANGE})
    got.update({"j=1": c["j=1"], "j=k-1, k>=3": c["j=k-1, k>=3"],
                "resting BUY": CLASS_COUNT[(key, "BUY")], "resting SELL": CLASS_COUNT[(key, "SELL")]})
    missing = [k for k, v in got.items() if v == 0]
    cert.check(f"dataset: {key} covers k = 2..6, 1..3 decoys, j = 1, j = k - 1 and both resting sides",
               wired and not missing, expected=want, observed=got, unit="count",
               detail=f"missing: {missing}" if missing else "", rule=R_DATA)
for key in LADDER_KEYS:
    c = COVER_SS[key]
    got = {f"k={k}": c["k", k] for k in K_RANGE}
    got.update({f"resting {sd}": c["resting", sd] for sd in ("BUY", "SELL")})
    missing = [k for k, v in got.items() if v == 0]
    cert.check(f"dataset: {key} same-second runs cover k = 2..6 and both resting sides", wired and not missing,
               expected={k: ">= 1" for k in got}, observed=got, unit="count",
               detail=f"missing: {missing}" if missing else "", rule=R_TS_SAME)
PAIR_CLASSES = [(PAIR, sd, sz) for sd in ("BUY", "SELL") for sz in ("q_agg<q_rest", "q_agg>q_rest")]
missing_pair = [c for c in PAIR_CLASSES if CLASS_COUNT.get(c, 0) == 0]
cert.check(f"dataset: {PAIR} covers aggressor BUY/SELL x q_agg < / > q_rest", wired and not missing_pair,
           expected={"/".join(c[1:]): ">= 1" for c in PAIR_CLASSES},
           observed={"/".join(c[1:]): CLASS_COUNT.get(c, 0) for c in PAIR_CLASSES}, unit="count",
           detail=f"missing: {missing_pair}" if missing_pair else "", rule=R_DATA)

# ---- eager: every required class and variant must have had one comparison -------------------------
REQUIRED_EAGER = ([(key, sd) for key in LADDER_KEYS for sd in ("BUY", "SELL")] + PAIR_CLASSES
                  + [(key, v) for key in LADDER_KEYS
                     for v in ("m1_reversal", "self_match") + TWIN_VARIANTS + SAME_SECOND_RUNS])
for tag in REQUIRED_EAGER:
    if tag not in EAGER_DONE:
        cert.set_eager_jit(None, False, detail=f"MISSING: {'/'.join(tag)} was never compared eager")


# ---- the per-run table: failing rows first, then the first 20 ---------------------------------------
def _fmt(r):
    e, o = r["e"], r["o"]
    g = same_second_groups(e)
    gs = g["same_price"] + g["across_prices"]
    same = "—" if not gs else ("yes" if group_ts_ok(o["ts"], gs) else "NO")
    return (f"| {r['key'].split('.')[1]} | {r['seed']} | {r['run']} | {len(e['touch'])} | "
            f"{e['filled_set']} | {o['filled_set']} | {[f[0] for f in e['fills']]} | {[f[0] for f in o['seq']]} | "
            f"{'yes' if o['book'] == e['book'] else 'NO'} | {'yes' if ts_ok(o['ts']) else 'NO'} | {same} | "
            f"{'PASS' if r['ok'] else 'FAIL'} |")


_hdr = ["| construction | seed | run | k (touch) | filled exp | filled obs | sequence exp | sequence obs | "
        "book ok | ts ok | same-second ok | row |", "|" + "---|" * 12]
_fail = [r for r in PER_RUN if not r["ok"]]
display(Markdown(f"**Per-run order ids** ({len(PER_RUN)} runs; {len(_fail)} failing)\n\n"
                 + "\n".join(_hdr + [_fmt(r) for r in _fail] + [_fmt(r) for r in PER_RUN[:20] if r["ok"]])))
print(f"valid draws: {VALID}; classes: {dict(CLASS_COUNT)}")
print(f"eager comparisons forced: {len(EAGER_DONE)} tags; "
      f"{sum(1 for a in cert.payload()['assertions'] if a['kind'] == 'eager_jit')} eager_jit assertions in all")
print(f"pre-run margins: largest hop bound {PEAK['hop_bound']} (<= {ENVELOPE.max_hops}), "
      f"largest ceil(qty/peak) {PEAK['max_slices']} (<= {SLICE_MARGIN})")

display(Markdown(cert.table()))

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))

## 9. Why this certification failed

### 9.1 What failed

800 assertion(s) failed, first 'C03.priority_ladder seed 0 same_second: stored timestamps unique and increasing in entry order' on seed 0: expected {'entry_order': [1, 2, 3, 4, 5, 6, 7, 8], 'stamps': 'one [sec, nsec] each, strictly increasing'}, observed [[1, [[1, 0]]], [2, [[1, 1]]], [3, [[1, 2]]], [4, [[1, 3]]], [5, [[1, 4]]], [6, [[1, 5]]], [7, [[1, 0]]], [8, [[1, 0]]]] (other). In full: all 800 failures are kind oracle, rule §5.2, and lie only on the r2 same-second runs. Two assertions fail there: (T) 'stored timestamps unique and increasing in entry order' and (T=) 'same-second stamps unique and sequential in entry order (across prices)'. They fail on both `same_second` and `same_second_reversed`, for each of the first 100 draws (seeds 0-99) of `C03.priority_ladder` and of `C03.price_over_time`: 2 assertions x 2 runs x 2 constructions x 100 seeds = 800. Every other assertion passes, on these runs and on every other run: the filled set (S), the counterparty sequence (Q), the book afterwards (B), (T=) among same-price orders, the M1 relations, M6, the battery RB and I1-I9, and eager = jitted. Stamps are stored as [SEC, NSEC] in integer seconds and integer sub-second units. Expected: the resting orders' stamps strictly increasing in entry order. Observed on ladder seed 0 `same_second` (touch oids 1-6, decoys 7 and 8, every row in tape second 1): oids 1-6 hold [1, 0], [1, 1], ..., [1, 5], and decoys 7 and 8 each hold [1, 0]. Observed on price-over-time seed 0 `same_second` (decoys 1 and 2 entered first, touch oids 3-8): decoys 1 and 2 each hold [1, 0], and touch oids 3-8 hold [1, 0], ..., [1, 5]. The pattern is the same in every failing run. At the touch, NSEC runs 0, 1, 2, ... in entry order. An order that rests alone at its price in that second holds [1, 0], which repeats the first touch order's stamp whatever its position in the entry order.

### 9.2 The smallest failing case

Seed 11, 4 real tape rows; frozen fixture `notebooks/certification/fixtures/C03_seed11_same_second.npz` (keys `tape`, `params_json`).

Smallest failing run: `C03.priority_ladder` seed 11, derived run `same_second` (k = 2, j = 1, one decoy; every row in tape second 1). Columns are integers: price_int (0.01 EUR/MWh offset encoding), qty_int (0.1 MW).

| row | role | oid | side | price_int (EUR/MWh) | qty_int | tape ts | expected stamp | observed stamp |
|---|---|---|---|---|---|---|---|---|
| 1 | touch T1 | 1 | SELL | 1006509 (66.09) | 25 | 1 | earliest | [1, 0] |
| 2 | touch T2 | 2 | SELL | 1006509 (66.09) | 25 | 1 | after oid 1 | [1, 1] |
| 3 | decoy D1 | 3 | SELL | 1006559 (66.59) | 25 | 1 | after oid 2 | **[1, 0]** (equals oid 1) |
| 4 | aggressor | 4 | BUY | 1006609 (67.09) | 25 | 1 | (not asserted) | - |

Fills, book and same-price stamps are as expected: oid 4 takes oid 1 for 25 at 1006509, and oids 2 and 3 rest whole. On `same_second_reversed` (T2 entered first) the stamps are oid 2 [1, 0], oid 1 [1, 1] and oid 3 [1, 0]. The K5-time fixture (`/tmp/omie_cert_fixtures_4n2kjvj2/C03_seed0.npz`) was a passing `same_owner_pair` tape, a harness artefact (see 9.4), so it is superseded by the fixture above.

### 9.3 Which rule it concerns

§5.2 (single global queue: unique, sequential); §13.1 (whole-second submission times). market-spec.md v2.14 §5.2: "Each order receives a timestamp when entering the SOB." "All timestamps across orders and explicit capacity requests are assigned from a single global queue (unique, sequential). Source: Continuous Trading Matching Algorithm §4.3." §13.1: "| 13 | Momento de envío | Submission time | datetime | `01/06/2025 17:41:12` | DD/MM/YYYY HH:MM:SS format." The regulatory source behind it is cited, not machine-checked. Continuous Trading Matching Algorithm §5.5: "all orders and all explicit capacity requests have unique timestamps, so that it is always possible to sort them by time stamp unequivocally." The failing clause is "single global queue (unique, sequential)", read across prices. §6.1's ranking ("ties broken by earliest timestamp") holds on every run.

### 9.4 Where the fault most plausibly lies

Hypothesis: the fault most plausibly lies in the simulator's stamp scheme. The cause is a deliberate scope limit of OMIE-STAMP-UNIQUE-1, not a regression. Whether that scope is conformant is a question of how §5.2 is read, and it is open for the human. It does NOT most plausibly lie in the judging code or the run. Evidence: (1) Code. `contract_book._rank_nsec` ranks NSEC only among live rows on the order's own side with `P == price_int` and `SEC == sec`. When no such peer exists it leaves the NSEC written at dispatch, which is 0. So an order that rests alone at its price in a second stores [sec, 0], whatever its position in the entry order. The observed stamps follow this rule exactly on all 400 failing runs: at the touch, NSEC runs 0, 1, 2, ... in entry order, and every lone-price decoy holds [1, 0]. The narrower scope was chosen on purpose. The card's title is 'stored entry stamps are unique among same-price, same-second resting orders', and its correct-behaviour clause reads 'Among same-price, same-second resting entries, the stamp is strictly increasing'. That clause was framed by C05's same-price collision. (2) Reading of the rule. §5.2 ('single global queue (unique, sequential)') and Matching Algorithm §5.5 ('all orders ... have unique timestamps, so that it is always possible to sort them by time stamp unequivocally') do not limit the queue to one price. So the K2 reading, a global queue with stamps unique across prices, is the literal one, and the per-price reading is narrower. The question for the human: is §5.2's uniqueness a property of the stored stamp? Then the simulator does not conform, and the fix is to rank NSEC among every same-SEC row (at least on the order's side, arguably on both sides). Or does it matter only through §6.1's tie-break? Then the across-price parts of (T) and (T=) over-assert, and the card should say so. (3) No behavioural effect is observed. Every ranking site compares price first. jaxob's `_get_top_bid_order_idx` and `_get_top_ask_order_idx` compare SEC and NSEC only among rows at the best price, and `iceberg_manager`'s ranked-stamp key is (±P, SEC, NSEC). So a collision across prices cannot change which order fills. Consistent with this, on all 400 failing runs (S), (Q), (B), the same-price (T=), M1 and the battery all pass, and the defect shows only in the stored field. A BUY and a SELL that rest in the same second can also share a stamp, because the helper reads its own side only. C03 has one resting side, so it does not exercise this. (4) Most plausibly not the judging code. The stamps are read from the book just before the aggressor and agree with the engine's rule to the unit. The same (T) assertion passes on every run whose rows have distinct tape seconds, and on the `shared_ts` twins, where only same-price touch orders share a second. Eager = jitted on every compared run. One judging-harness artefact was found, and it does not touch the verdict. At K5 the frozen fixture (`/tmp/omie_cert_fixtures_4n2kjvj2/C03_seed0.npz`, 2 rows) was the `C03.same_owner_pair` tape, which passes. `Certification.record_case` keys cases by seed alone, so the last construction recorded for seed 0 overwrote the others, and the fixture was the base draw, not the derived run. Section 9.2 now freezes the failing derived tape. (5) Most plausibly not the run. Failures are deterministic: 100/100 same-second draws fail in both constructions and in both entry orders, and no other run fails. There are no discards (1100/1100 valid). Discriminating probe: replay the 9.2 fixture. Under the global reading, decoy oid 3 must store a stamp after oid 2's [1, 1], for example [1, 2]. Under the per-price reading, [1, 0] is correct and the across-price assertions should be withdrawn

### 9.5 What it affects downstream

C05: its same-second residual and interloper share one price, so the per-price rank covers them. A stored-stamp uniqueness assertion read globally would raise this same question. C08 and C09: regenerated iceberg slices take stamps from the counter, while tape orders take (second, rank). Only per-price uniqueness is guaranteed, so a slice at a drifted price and a tape order at another price can share a stamp. C15: P3 (stored-stamp uniqueness) at scale. Random sessions with same-second arrivals at different prices, or on opposite sides, will show this collision wherever P3 is read globally. C04 and C06: these rely on queue order within a price and on price-first level order. Neither depends on uniqueness across prices, so neither should be affected. C13: affected only if an observation exposes stored stamps. Track B: passive-fill figures are unaffected, because priority is decided within a price. Any feature, log or analysis that sorts events across prices by stored stamp would see ties
